# 06 - Experimental Matrix Audit & Monitoring Dashboard (Dynamic DB-Driven)

**Scientific & Operational Objectives:**
1. **Dynamic Database Discovery**: Directly queries  and  to detect dimensions, noise regimes, problem IDs, and model architectures without hardcoded lists.
2. **Two-Tier Pipeline Audit**:
   - **Tier 1 (Evolutionary Phase - SQLite)**: Audits LLaMEA evolutionary synthesis runs, iteration counts, convergence frequencies, and database integrity.
   - **Tier 2 (Evaluation Phase - IOH Logs)**: Audits post-evolution =10$ benchmark evaluations for LLM champions and classical baselines (, , ).
3. **Sample Imbalance & Gap Detection**: Quantifies run distribution imbalances and flags unexecuted experimental cells.
4. **Automated Report & Visual Export**:
   - Generates 
   - Renders 
5. **Actionable Task Dispatcher**: Emits ready-to-run commands for any missing experimental condition.


In [23]:
import sys
import os
import re
import sqlite3
import json
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from types import SimpleNamespace
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from core.config import DATA_DIR, PROJECT_ROOT, RESULTS_DIR

DB_PATH = DATA_DIR / 'db.sqlite3'
EVALUATIONS_DIR = RESULTS_DIR / 'evaluations'
REPORTS_DIR = RESULTS_DIR / 'reports'
FIGURES_DIR = RESULTS_DIR / 'figures'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Dynamic BBOB Metadata (All 24 Benchmark Functions) ───────────────────
BBOB_METADATA = {
    1:  ("Sphere", "Separable"),
    2:  ("Ellipsoidal", "Separable"),
    3:  ("Rastrigin", "Separable"),
    4:  ("Buche-Rastrigin", "Separable"),
    5:  ("Linear Slope", "Separable"),
    6:  ("Attractive Sector", "Low Conditioning"),
    7:  ("Step Ellipsoidal", "Low Conditioning"),
    8:  ("Rosenbrock", "Low Conditioning"),
    9:  ("Rosenbrock Rotated", "Low Conditioning"),
    10: ("Ellipsoidal High-Cond", "High Conditioning"),
    11: ("Discus", "High Conditioning"),
    12: ("Bent Cigar", "High Conditioning"),
    13: ("Sharp Ridge", "High Conditioning"),
    14: ("Different Powers", "High Conditioning"),
    15: ("Rastrigin Multi-Modal", "Multi-Modal (Global)"),
    16: ("Weierstrass", "Multi-Modal (Global)"),
    17: ("Schaffers F7", "Multi-Modal (Global)"),
    18: ("Schaffers F7 Ill-Cond", "Multi-Modal (Global)"),
    19: ("Griewank-Rosenbrock", "Multi-Modal (Global)"),
    20: ("Schwefel", "Multi-Modal (Weak)"),
    21: ("Gallagher 101 Peaks", "Multi-Modal (Weak)"),
    22: ("Gallagher 21 Peaks", "Multi-Modal (Weak)"),
    23: ("Katsuura", "Multi-Modal (Weak)"),
    24: ("Lunacek Bi-Rastrigin", "Multi-Modal (Weak)")
}

def get_bbob_name(p_id: int) -> str:
    name, _ = BBOB_METADATA.get(p_id, (f"Function {p_id}", "General"))
    return f"{name} (f{p_id})"

def get_bbob_class(p_id: int) -> str:
    _, cls = BBOB_METADATA.get(p_id, (f"Function {p_id}", "General"))
    return cls

BBOB_NAMES_MAP = {p: get_bbob_name(p) for p in range(1, 25)}
BBOB_CLASSES_MAP = {p: get_bbob_class(p) for p in range(1, 25)}
BBOB_NAMES = BBOB_NAMES_MAP
BBOB_CLASSES = BBOB_CLASSES_MAP

def get_clean_model_label(llm_name: str) -> str:
    l = str(llm_name).lower()
    m = re.search(r'(\d+b)', l)
    if m:
        return f"LLaMEA-{m.group(1).upper()}"
    clean = l.removesuffix('.gguf').replace('-', '_').split('/')[-1]
    return f"LLaMEA-{clean.title()}"

def map_db_solver_name(llm_name: str, strat: str) -> str:
    model_lbl = get_clean_model_label(llm_name)
    return f"{model_lbl} / {str(strat).lower()}"

def resolve_folder_solver_name(folder_name: str) -> str:
    p = re.sub(r'(-\d+|\.\d+)$', '', folder_name.strip().lower())
    if p in ['cmaes', 'cma_es', 'cma-es']: return 'CMA-ES'
    if p == 'de' or p.startswith(('de_', 'de-')) or '_de_' in p: return 'DE'
    if p == 'pso': return 'PSO'
    
    strategies = ['baseline', 'guided', 'thinking', 'vectorization']
    for s in strategies:
        if p.endswith(f'_{s}') or p.endswith(f'-{s}'):
            model_part = p[: -(len(s) + 1)]
            m = re.search(r'(\d+b)', model_part)
            if m:
                return f"LLaMEA-{m.group(1).upper()} / {s}"
            clean_m = model_part.replace('llamea', '').strip('_-').replace('_', ' ').title()
            return f"LLaMEA-{clean_m} / {s}" if clean_m else f"LLaMEA / {s}"
    
    return folder_name

def load_audit_data(db_path: Path, eval_dir: Path) -> SimpleNamespace:
    df_exp, df_iter = pd.DataFrame(), pd.DataFrame()
    eval_counts = defaultdict(lambda: defaultdict(int))
    dims, noise_levels, problem_ids, db_solvers = [], [], [], set()

    if db_path.exists():
        with sqlite3.connect(db_path) as conn:
            df_exp = pd.read_sql_query("SELECT * FROM experiments WHERE status = 'completed';", conn)
            df_iter = pd.read_sql_query(
                "SELECT i.id AS iteration_id, i.experiment_id, i.algorithm_name, "
                "i.raw_fitness, i.final_error, i.timed_out, i.converged, i.runtime_seconds, "
                "e.problem_id, e.dim, e.mode, e.llm_name, e.prompt_strategy, e.noise_std "
                "FROM iterations i JOIN experiments e ON i.experiment_id = e.id WHERE e.status = 'completed';", conn
            )
        if not df_exp.empty:
            df_exp["solver_name"] = df_exp.apply(
                lambda r: map_db_solver_name(r["llm_name"], r["prompt_strategy"]), axis=1
            )
            dims = sorted(df_exp["dim"].unique().tolist())
            noise_levels = sorted(df_exp["noise_std"].unique().tolist())
            problem_ids = sorted(df_exp["problem_id"].unique().tolist())
            db_solvers = set(df_exp["solver_name"].unique())

    if eval_dir.exists():
        # Iterate over all solver directories: {dim}D/std_{noise}/f{p_id}/{solver_folder}
        for solver_dir in eval_dir.glob("*/*/*/*"):
            if not solver_dir.is_dir():
                continue
            
            parts = solver_dir.relative_to(eval_dir).parts
            if len(parts) < 4:
                continue
            
            dim_str, noise_str, p_str, solver_folder = parts[0], parts[1], parts[2], parts[3]
            dim_m = re.search(r"(\d+)D", dim_str)
            dim = int(dim_m.group(1)) if dim_m else None
            noise_m = re.search(r"std_([\d\.]+)", noise_str)
            noise_std = float(noise_m.group(1)) if noise_m else 0.0
            p_m = re.search(r"f(\d+)", p_str)
            p_id = int(p_m.group(1)) if p_m else None

            solver = resolve_folder_solver_name(solver_folder)
            dat_files = [f for f in solver_dir.glob("**/*.dat") if f.stat().st_size > 0]
            if not dat_files:
                continue
            
            # Count exact number of runs
            n_runs = 0
            json_files = list(solver_dir.glob("**/*.json"))
            ioh_jsons = [f for f in json_files if "IOHprofiler" in f.name]
            if ioh_jsons:
                try:
                    with open(ioh_jsons[0]) as jf:
                        meta = json.load(jf)
                    runs = meta.get("scenarios", [{}])[0].get("runs", [])
                    n_runs = len(runs) if runs else 10
                except Exception:
                    n_runs = 10
            elif (solver_dir / "provenance.json").exists():
                try:
                    with open(solver_dir / "provenance.json") as pf:
                        prov = json.load(pf)
                    n_runs = int(prov.get("n_runs", 10))
                except Exception:
                    n_runs = 10
            else:
                n_runs = 10

            eval_counts[(dim, noise_std, p_id)][solver] = max(eval_counts[(dim, noise_std, p_id)][solver], n_runs)

    dims = sorted(set(dims))
    noise_levels = sorted(set(noise_levels))
    problem_ids = sorted(set(problem_ids))
    
    # Discover all unique solvers across DB and Evaluation files
    all_eval_solvers = {s for cond in eval_counts.values() for s in cond}
    all_solvers_union = db_solvers.union(all_eval_solvers)
    
    classical = [s for s in ['CMA-ES', 'DE', 'PSO'] if s in all_solvers_union]
    other_classical = sorted([s for s in all_solvers_union if not s.startswith('LLaMEA') and s not in classical])
    
    # Sort LLM models by model name and strategy
    llm_solvers = sorted([s for s in all_solvers_union if s.startswith('LLaMEA')])
    model_families = sorted(list({s.split(' / ')[0] for s in llm_solvers}))
    all_solvers_ordered = llm_solvers + classical + other_classical

    return SimpleNamespace(
        db_path=db_path, eval_dir=eval_dir, df_exp=df_exp, df_iter=df_iter,
        eval_counts=eval_counts, dims=dims, noise_levels=noise_levels,
        problem_ids=problem_ids, llm_solvers=llm_solvers,
        model_families=model_families,
        classical_solvers=classical + other_classical,
        all_solvers=all_solvers_ordered
    )


In [24]:
loader = load_audit_data(DB_PATH, EVALUATIONS_DIR)
print("🔍 Dynamic DB & Model Discovery Summary:")
print(f"   • Dimensions ({len(loader.dims)}): {loader.dims}")
print(f"   • Noise Levels ({len(loader.noise_levels)}): {loader.noise_levels}")
print(f"   • Problem IDs ({len(loader.problem_ids)}): {loader.problem_ids}")
print(f"   • Discovered Solvers ({len(loader.all_solvers)}):")
for s in loader.all_solvers:
    print(f"       - {s}")


🔍 Dynamic DB & Model Discovery Summary:
   • Dimensions (3): [2, 3, 5]
   • Noise Levels (2): [0.0, 0.05]
   • Problem IDs (5): [1, 8, 11, 15, 21]
   • Discovered Solvers (10):
       - LLaMEA-14B / baseline
       - LLaMEA-14B / guided
       - LLaMEA-14B / thinking
       - LLaMEA-14B / vectorization
       - LLaMEA-7B / baseline
       - LLaMEA-7B / guided
       - LLaMEA-7B / thinking
       - LLaMEA-7B / vectorization
       - CMA-ES
       - DE


In [25]:
# ── 3. Compile Multi-Tier Experimental Audit DataFrames ───────────────────
eval_records = []
for d in loader.dims:
    for n in loader.noise_levels:
        for p in loader.problem_ids:
            for s in loader.all_solvers:
                cnt = loader.eval_counts[(d, n, p)][s]
                eval_records.append({
                    "Dimension": f"{d}D",
                    "Noise": f"σ={n}",
                    "Problem_ID": p,
                    "Problem_Name": BBOB_NAMES_MAP.get(p, f"f{p}"),
                    "Hardness_Class": BBOB_CLASSES_MAP.get(p, "Unknown"),
                    "Solver": s,
                    "Evaluated_Runs": cnt,
                    "Status": "Complete" if cnt >= 10 else ("Partial" if cnt > 0 else "Missing")
                })

df_eval_audit = pd.DataFrame(eval_records)
total_cells = len(df_eval_audit)
completed_cells = len(df_eval_audit[df_eval_audit["Evaluated_Runs"] > 0])
missing_cells = len(df_eval_audit[df_eval_audit["Evaluated_Runs"] == 0])
completion_pct = (completed_cells / total_cells) * 100.0 if total_cells > 0 else 0.0

# SQLite Evolutionary Phase Audit
if not loader.df_exp.empty:
    db_summary = loader.df_exp.groupby(["dim", "noise_std", "problem_id", "solver_name"]).size().reset_index(name="experiment_count")
    print(f"🧬 SQLite Evolutionary Experiments Logged: {len(loader.df_exp)} across {len(loader.df_iter)} iterations.")

print(f"🎯 Benchmark Evaluation Matrix (IOH Logs): {completed_cells}/{total_cells} cells ({completion_pct:.1f}% complete).")


🧬 SQLite Evolutionary Experiments Logged: 292 across 2902 iterations.
🎯 Benchmark Evaluation Matrix (IOH Logs): 148/300 cells (49.3% complete).


In [26]:
# ── 4. Dynamic Markdown Audit Report Generator ───────────────────────────
missing_cells_df = df_eval_audit[df_eval_audit['Evaluated_Runs'] == 0]
partial_cells_df = df_eval_audit[(df_eval_audit['Evaluated_Runs'] > 0) & (df_eval_audit['Evaluated_Runs'] < 10)]
target_cells_df  = df_eval_audit[df_eval_audit['Evaluated_Runs'] == 10]
over_cells_df    = df_eval_audit[df_eval_audit['Evaluated_Runs'] > 10]

n_total = len(df_eval_audit)
n_miss = len(missing_cells_df)
n_part = len(partial_cells_df)
n_target = len(target_cells_df)
n_over = len(over_cells_df)

report_lines = []
report_lines.append('# 📋 Experimental Matrix Coverage & Sample Imbalance Audit Report\n')
report_lines.append(f'**Generated:** `notebooks/06_experimental_audit.ipynb` | {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
report_lines.append('### 📊 High-Level Status Breakdown')
report_lines.append(f'- **Total Experimental Cells Planned:** `{n_total}`')
report_lines.append(f'- 🔴 **Missing Conditions (0 Runs):** `{n_miss}` ({n_miss/n_total*100:.1f}%)')
report_lines.append(f'- 🟡 **Partial / Interrupted Runs (<10 Runs):** `{n_part}` ({n_part/n_total*100:.1f}%)')
report_lines.append(f'- 🟢 **Exact Target Met (N = 10 Runs):** `{n_target}` ({n_target/n_total*100:.1f}%)')
report_lines.append(f'- 🔵 **Over-Sampled / Imbalanced (N > 10 Runs):** `{n_over}` ({n_over/n_total*100:.1f}%)\n')
report_lines.append('---\n')

report_lines.append('## 1. Completion Rate by Dimension\n')
report_lines.append('| Dimension | Total Cells | Fully Completed (N ≥ 10) | Partial (<10) | Missing (0) | Completion Rate |')
report_lines.append('|---|---|---|---|---|---|')
for d in loader.dims:
    sub = df_eval_audit[df_eval_audit['Dimension'] == f'{d}D']
    tot = len(sub)
    full = len(sub[sub['Evaluated_Runs'] >= 10])
    part = len(sub[(sub['Evaluated_Runs'] > 0) & (sub['Evaluated_Runs'] < 10)])
    miss = len(sub[sub['Evaluated_Runs'] == 0])
    rate = ((full + part) / tot) * 100.0 if tot > 0 else 0.0
    report_lines.append(f'| **{d}D** | {tot} | {full} | {part} | {miss} | {rate:.1f}% |')
report_lines.append('\n---\n')

report_lines.append('## 2. Sample Size Imbalance by Solver\n')
report_lines.append('| Solver | Category | Evaluated Cells | Missing Cells | Mean Runs (N) | Min Runs | Max Runs |')
report_lines.append('|---|---|---|---|---|---|---|')
for s in loader.all_solvers:
    sub = df_eval_audit[df_eval_audit['Solver'] == s]
    tot = len(sub)
    comp = len(sub[sub['Evaluated_Runs'] > 0])
    miss = len(sub[sub['Evaluated_Runs'] == 0])
    runs_s = sub[sub['Evaluated_Runs'] > 0]['Evaluated_Runs']
    mean_n = runs_s.mean() if comp > 0 else 0.0
    min_n = runs_s.min() if comp > 0 else 0
    max_n = runs_s.max() if comp > 0 else 0
    arch = 'LLaMEA-14B' if '14B' in s else ('LLaMEA-7B' if '7B' in s else 'Classical Baseline')
    report_lines.append(f'| **{s}** | {arch} | {comp}/{tot} | {miss} | **{mean_n:.1f}** | {min_n} | {max_n} |')
report_lines.append('\n---\n')

report_lines.append('## 3. Actionable Checklist: Missing Experiments (0 Runs)\n')
if missing_cells_df.empty:
    report_lines.append('🎉 **No missing conditions! Full coverage achieved.**\n')
else:
    report_lines.append('| # | Dimension | Noise Regime | Problem Name | Problem Class | Solver | Action Required |')
    report_lines.append('|---|---|---|---|---|---|---|')
    for idx, (_, row) in enumerate(missing_cells_df.iterrows(), 1):
        report_lines.append(f'| {idx} | {row["Dimension"]} | {row["Noise"]} | {row["Problem_Name"]} | {row["Hardness_Class"]} | **{row["Solver"]}** | 🔴 **Execute N=10 Runs** |')
report_lines.append('\n---\n')

report_lines.append('## 4. Actionable Checklist: Partial / Interrupted Experiments (< 10 Runs)\n')
if partial_cells_df.empty:
    report_lines.append('🎉 **No partial runs detected!**\n')
else:
    report_lines.append('| # | Dimension | Noise Regime | Problem Name | Solver | Completed Runs | Remaining to Target (10 - N) |')
    report_lines.append('|---|---|---|---|---|---|---|')
    for idx, (_, row) in enumerate(partial_cells_df.iterrows(), 1):
        rem = 10 - int(row['Evaluated_Runs'])
        report_lines.append(f'| {idx} | {row["Dimension"]} | {row["Noise"]} | {row["Problem_Name"]} | **{row["Solver"]}** | ⚠️ {row["Evaluated_Runs"]}/10 | 🟡 **Run +{rem} more** |')

report_path = REPORTS_DIR / 'experimental_coverage_audit.md'
with open(report_path, 'w') as f:
    f.write('\n'.join(report_lines))
print(f'✅ Dynamic Audit Report written to: {report_path}')


✅ Dynamic Audit Report written to: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/reports/experimental_coverage_audit.md


In [27]:
# ── 5. Render High-Contrast Status & Imbalance Matrices Per LLM Model ───
def render_matrix_for_model(loader, model_family: str):
    slug = model_family.lower().replace(" ", "_").replace("-", "_")
    solvers = [s for s in loader.llm_solvers if s.startswith(model_family)] + loader.classical_solvers
    
    n_rows = len(loader.dims)
    n_cols = len(loader.noise_levels)

    def get_status_code(cnt):
        if cnt == 0: return 0.0      # Missing (Red)
        elif cnt < 10: return 1.0    # Partial (Amber)
        elif cnt == 10: return 2.0   # Target Met (Green)
        else: return 3.0             # Over-Sampled / Imbalance (Blue)

    def get_status_text(cnt):
        if cnt == 0: return '❌ 0'
        elif cnt < 10: return f'⚠️ {cnt}/10'
        elif cnt == 10: return '✅ 10'
        else: return f'🔷 {cnt}'

    colorscale = [
        [0.00, '#FFCDD2'], [0.24, '#FFCDD2'], # Missing (Soft Red)
        [0.26, '#FFE082'], [0.49, '#FFE082'], # Partial (Soft Amber)
        [0.51, '#C8E6C9'], [0.74, '#C8E6C9'], # Target Met (Soft Green)
        [0.76, '#BBDEFB'], [1.00, '#BBDEFB']  # Over-Sampled / Imbalance (Soft Blue)
    ]

    subplot_titles = []
    coords_map = {}
    for r_idx, d in enumerate(loader.dims, 1):
        for c_idx, n in enumerate(loader.noise_levels, 1):
            label = 'Clean (σ=0.0)' if n == 0.0 else f'Noisy (σ={n})'
            total = len(solvers) * len(loader.problem_ids)
            done = sum(1 for s in solvers for p in loader.problem_ids if loader.eval_counts[(d, n, p)][s] > 0)
            pct = (done / total) * 100 if total > 0 else 0
            subplot_titles.append(f'<b>{d}D — {label}</b>  <span style="font-size:11px; color:#555;">({done}/{total} done · {pct:.0f}%)</span>')
            coords_map[(d, n)] = (r_idx, c_idx)

    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.10,
        vertical_spacing=0.10
    )

    prob_labels = [BBOB_NAMES_MAP.get(p, f'f{p}') for p in loader.problem_ids]

    for d in loader.dims:
        for n in loader.noise_levels:
            r, c = coords_map[(d, n)]
            z_vals, text_vals = [], []
            for s in solvers:
                row_z, row_t = [], []
                for p in loader.problem_ids:
                    cnt = loader.eval_counts[(d, n, p)][s]
                    row_z.append(get_status_code(cnt))
                    row_t.append(get_status_text(cnt))
                z_vals.append(row_z)
                text_vals.append(row_t)
                
            fig.add_trace(
                go.Heatmap(
                    z=z_vals,
                    x=prob_labels,
                    y=solvers,
                    text=text_vals,
                    texttemplate='<b>%{text}</b>',
                    textfont=dict(size=11, family='Inter, Helvetica, Arial, sans-serif', color='#1E293B'),
                    colorscale=colorscale,
                    zmin=0.0, zmax=3.0,
                    showscale=False,
                    xgap=3, ygap=3
                ),
                row=r, col=c
            )
            if c == 1:
                fig.update_yaxes(autorange='reversed', row=r, col=c, tickfont=dict(size=11, family='Inter, sans-serif'))
            else:
                fig.update_yaxes(showticklabels=False, autorange='reversed', row=r, col=c)
            fig.update_xaxes(tickangle=-20, row=r, col=c, tickfont=dict(size=10, family='Inter, sans-serif'))

    fig.update_layout(
        template='plotly_white',
        title=dict(
            text=f'<b>Experimental Coverage & Sample Imbalance — {model_family} vs Baselines</b><br>' +
                 '<sup><b>Legend:</b>  ' +
                 '<span style="color:#D32F2F;">■</span> <b>❌ 0 (Missing)</b>   &nbsp;|&nbsp;   ' +
                 '<span style="color:#F57C00;">■</span> <b>⚠️ &lt;10 (Partial)</b>   &nbsp;|&nbsp;   ' +
                 '<span style="color:#2E7D32;">■</span> <b>✅ 10 (Target Met)</b>   &nbsp;|&nbsp;   ' +
                 '<span style="color:#1976D2;">■</span> <b>🔷 &gt;10 (Over-Sampled / Imbalance)</b></sup>',
            x=0.02, y=0.98,
            font=dict(size=13, color='#0F172A', family='Inter, Helvetica, Arial, sans-serif')
        ),
        width=1240, height=280 * n_rows + 40,
        margin=dict(l=190, r=40, t=110, b=50),
        font=dict(family='Inter, Helvetica, Arial, sans-serif', size=11, color='#0F172A')
    )

    out_p = FIGURES_DIR / f'experimental_coverage_matrix_{slug}.png'
    fig.write_image(str(out_p), scale=3)
    print(f'✅ High-legibility status matrix exported to: {out_p}')
    return fig

# Dynamically generate and export a matrix for every discovered LLM model family
generated_figures = {}
for mf in loader.model_families:
    generated_figures[mf] = render_matrix_for_model(loader, mf)


✅ High-legibility status matrix exported to: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/experimental_coverage_matrix_llamea_14b.png
✅ High-legibility status matrix exported to: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/experimental_coverage_matrix_llamea_7b.png


In [28]:
# ── 6. Missing Experiment Action Plan & CLI Dispatcher ───────────────────
missing_df = df_eval_audit[df_eval_audit["Evaluated_Runs"] == 0]
if not missing_df.empty:
    print(f"⚠️ {len(missing_df)} Experimental Conditions Require Execution:")
    for _, r in missing_df.iterrows():
        print(f"  • Dimension {r['Dimension']} | {r['Noise']} | {r['Problem_Name']} | Solver: {r['Solver']}")
else:
    print("🎉 All experimental matrix conditions are completely evaluated!")


⚠️ 152 Experimental Conditions Require Execution:
  • Dimension 2D | σ=0.0 | Sphere (f1) | Solver: LLaMEA-7B / baseline
  • Dimension 2D | σ=0.0 | Sphere (f1) | Solver: LLaMEA-7B / guided
  • Dimension 2D | σ=0.0 | Sphere (f1) | Solver: LLaMEA-7B / thinking
  • Dimension 2D | σ=0.0 | Sphere (f1) | Solver: LLaMEA-7B / vectorization
  • Dimension 2D | σ=0.0 | Rosenbrock (f8) | Solver: LLaMEA-7B / baseline
  • Dimension 2D | σ=0.0 | Rosenbrock (f8) | Solver: LLaMEA-7B / guided
  • Dimension 2D | σ=0.0 | Rosenbrock (f8) | Solver: LLaMEA-7B / thinking
  • Dimension 2D | σ=0.0 | Rosenbrock (f8) | Solver: LLaMEA-7B / vectorization
  • Dimension 2D | σ=0.0 | Discus (f11) | Solver: LLaMEA-7B / baseline
  • Dimension 2D | σ=0.0 | Discus (f11) | Solver: LLaMEA-7B / guided
  • Dimension 2D | σ=0.0 | Discus (f11) | Solver: LLaMEA-7B / thinking
  • Dimension 2D | σ=0.0 | Discus (f11) | Solver: LLaMEA-7B / vectorization
  • Dimension 2D | σ=0.0 | Rastrigin Multi-Modal (f15) | Solver: LLaMEA-7B / base